# 🗓️ 26일차 스터디 노트북 — 브루트 포스법 (문자열 검색의 시작)

**오늘 범위**: 07장 시작 · 07-1 브루트 포스법 — 문자열 검색이란 · 검색 과정 손추적 · 실습 7-1 `bf_match` · 보충수업 7-1 (멤버십 연산자, find/index 계열, with 계열)

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[손]** · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]** · **[추적]**

---

## 06장을 닫고 07장을 연다

지난 8일간(18~25일차) **정렬 8종**을 전부 손으로 짰어. 오늘부터는 **문자열 검색**이야.

교재 305p: **"문자열에서 부분 문자열을 검색하는 알고리즘으로 브루트 포스법, KMP법, 보이어·무어법 등이 있습니다."**

| 절 | 알고리즘 | 특징 |
|---|---|---|
| **07-1** | **브루트 포스법 (오늘)** | 가장 단순. 실패하면 **1칸씩** 민다 |
| 07-2 | KMP법 | 실패해도 **이미 맞춘 정보를 기억**해서 여러 칸 민다 |
| 07-3 | 보이어·무어법 | 패턴을 **뒤에서부터** 비교. 실무 최강 |

교재 306p가 브루트 포스의 약점을 미리 찍어놨어:
> **"이렇게 이미 검사한 위치를 기억하지 못하므로 브루트 포스법은 효율이 좋지 않습니다."**

→ 이 한 문장이 07-2 KMP가 존재하는 이유 전부야. 오늘 그 "기억하지 못함"이 코드 어디에 있는지 정확히 짚어낼 거야.

## 오늘의 핵심 질문

> **`return pt - pp` 가 왜 시작 인덱스가 되는가?** 🔥

이게 오늘 노트북의 심장이야 (9~11번).

## 진행 순서
**개념(1~2) → 손 추적(3~5) → 불변식(6~8) → 코드 구현(9~12) → 파이썬 표준(13~17)**

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
import time, random, string

# 교재 실습 7-1
def bf_match(txt: str, pat: str) -> int:
    """브루트 포스법으로 문자열 검색"""
    pt = 0        # txt를 따라가는 커서
    pp = 0        # pat를 따라가는 커서

    while pt != len(txt) and pp != len(pat):
        if txt[pt] == pat[pp]:
            pt += 1
            pp += 1
        else:
            pt = pt - pp + 1
            pp = 0

    return pt - pp if pp == len(pat) else -1


def show_align(txt, pat, start, upto=None):
    """텍스트와 패턴을 위아래로 나란히 그린다 (교재 그림 7-3 스타일)"""
    idx = " ".join(f"{i%10}" for i in range(len(txt)))
    print(f"  idx  {idx}")
    print(f"  txt  {' '.join(txt)}")
    pad = "  " * start
    if upto is None:
        print(f"  pat  {pad}{' '.join(pat)}")
    else:
        shown = " ".join(pat[:upto+1])
        print(f"  pat  {pad}{shown}")

# 검산용 (아주 단순한 참조 구현)
def naive(t, p):
    for i in range(len(t) - len(p) + 1):
        if t[i:i+len(p)] == p:
            return i
    return -1

TXT = 'ABABCDEFGHA'
PAT = 'ABC'

print("준비 완료 ✅")
show_align(TXT, PAT, 0)
print(f"\n  len(txt) = {len(TXT)}, len(pat) = {len(PAT)}")

---
# 🔁 [Remind] 워밍업 — 되감기

07장의 첫날이지만, 오늘 쓰는 도구는 전부 6장에서 나왔어.

### R-1. 🟢 [설명] 커서 두 개는 처음이 아니야

```python
pt = 0    # txt를 따라가는 커서
pp = 0    # pat를 따라가는 커서
```

이번 주에 커서 두 개짜리 알고리즘을 여러 번 봤지.

| 알고리즘 | 커서들 | 움직임 |
|---|---|---|
| 셰이커 정렬 (18일) | 왼쪽/오른쪽 끝 | 서로 마주보며 좁혀짐 |
| 퀵 정렬 (21일) | `pl`, `pr` | 마주보며 좁혀짐 |
| 병합 (23일) | `pa`, `pb`, `pc` | 각자 전진 |
| **브루트 포스 (오늘)** | `pt`, `pp` | ①____ |

- ①을 채워봐. `pt`와 `pp`는 **같이 움직일까, 따로 움직일까?** 두 경우가 다 있지?
- 병합 정렬의 `pa`/`pb`는 **매 반복 둘 중 하나만** 증가했어. 오늘은?

### R-2. 🟡 [설명] 12일차 선형 검색의 확장

교재 305p: **"브루트 포스법은 선형 검색을 단순하게 확장한 알고리즘이라서 단순법이라고 합니다."**

12일차 선형 검색은 이랬어:
```python
for i in range(len(a)):
    if a[i] == key: return i
return -1
```

- 선형 검색은 배열의 각 자리에서 **값 1개**를 비교했어. 문자열 검색은 각 자리에서 **몇 개**를 비교하지?
- 그래서 시간 복잡도가 O(n)에서 O(?)로 바뀔까? 예측만 해두고 8번에서 확인해.
- 선형 검색의 "못 찾으면 `-1`" 규약이 오늘도 똑같이 쓰여. 코드 어디에 있는지 찾아봐.

*(여기에 답 작성)*

---
# 🎯 PART 1 — 문자열 검색이란 (1~2번)

### 1. 🟢 [설명] 텍스트와 패턴

교재 305p: **"문자열 검색은 어떤 문자열 안에 다른 문자열이 포함되어 있는지 검사하고, 만약 포함되어 있다면 어디에 위치하는지 찾아내는 것을 말합니다."**
그리고 **"검색되는 쪽의 문자열을 텍스트, 찾아내는 문자열을 패턴이라고 합니다."**

- `'STRING'`에서 `'IN'`을 찾으면 성공, `'QUEEN'`에서 `'IN'`을 찾으면 실패. 각각 왜 그런지 한 문장씩.
- **텍스트**와 **패턴** 중 보통 어느 쪽이 길까? 둘의 길이를 각각 `n`, `m`이라 할 때 `n`과 `m` 중 어느 게 커?
- 이 검색은 **두 가지 질문**에 답해야 해:
  - 질문 1: ①________________
  - 질문 2: ②________________
- 나중에 볼 `in` 연산자는 이 중 **어느 질문에만** 답할까? (13번에서 확인)

### 2. 🟢 [설명] "브루트 포스"라는 이름

**brute force** = 무차별 대입, 힘으로 밀어붙이기.

- 이 알고리즘이 왜 그런 이름을 얻었을까? 교재 305~306p의 검색 과정을 보고 한 문장으로.
- 교재 306p의 결정적 문장:
  > **"[그림 7-3]의 ③에서는 텍스트 쪽의 검사 위치가 ②까지 나아갔지만 그림 ④에서는 ①로 되돌아갑니다. 이렇게 이미 검사한 위치를 기억하지 못하므로 브루트 포스법은 효율이 좋지 않습니다."**
  
  → **"되돌아간다"** 는 게 코드의 어느 줄이야? (`bf_match` 안에서 찾아봐)
- 💡 이 "되돌아감"을 없애는 게 07-2 KMP야. 오늘은 **되돌아가는 걸 정확히 관찰**하는 게 목표야.

*(여기에 답 작성)*

---
# ✍️ PART 2 — 손으로 추적하기 (3~5번)

> 교재 306p [그림 7-3]을 **직접 그린다.** 오늘 노트북의 기본기야.

### 3. 🟢 [손] 그림 7-3 완전 재현

```
       0 1 2 3 4 5 6 7 8 9 10
  txt  A B A B C D E F G H A
  pat  A B C
```

교재는 이 검색을 **7단계**로 나눠 그렸어. 표를 직접 채워봐.

| 단계 | pt | pp | txt[pt] | pat[pp] | 일치? | 다음 동작 |
|---|---|---|---|---|---|---|
| ① | 0 | 0 | A | A | ✅ | pt=1, pp=1 |
| ② | 1 | 1 | B | B | ✅ | pt=2, pp=2 |
| ③ | 2 | 2 | ① | ② | ❌ | `pt = 2-2+1 = 1`, `pp = 0` |
| ④ | 1 | 0 | ③ | ④ | ⑤ | ⑥ |
| ⑤ | ⑦ | ⑧ | ⑨ | ⑩ | ⑪ | ⑫ |
| ⑥ | | | | | | |
| ⑦ | | | | | | |

**종료 시점**: `pt = ____`, `pp = ____` → `return ____`

- 교재 그림에서 ③ → ④로 갈 때 `pt`가 **2에서 1로 되돌아가지.** 이게 2번에서 말한 "기억하지 못함"이야.
- 총 몇 번 비교했어? 텍스트 길이 11, 패턴 길이 3인데 그것보다 많아, 적어?

*(끝까지 채운 뒤 실행)*

In [ ]:
txt, pat = TXT, PAT
pt = pp = 0
step = 0
print(f"txt = {txt}, pat = {pat}\n")
while pt != len(txt) and pp != len(pat):
    step += 1
    matched = txt[pt] == pat[pp]
    print(f"[{step}] pt={pt}, pp={pp}")
    show_align(txt, pat, pt - pp, pp)
    print(f"     txt[{pt}]='{txt[pt]}' vs pat[{pp}]='{pat[pp]}' → {'✅ 일치' if matched else '❌ 불일치'}")
    if matched:
        pt += 1; pp += 1
        print(f"     → pt={pt}, pp={pp} (둘 다 +1)\n")
    else:
        old_pt, old_pp = pt, pp
        pt = pt - pp + 1; pp = 0
        print(f"     → pt = {old_pt}-{old_pp}+1 = {pt}, pp = 0  (패턴을 1칸 밀기)\n")

print(f"종료: pt={pt}, pp={pp}, pp == len(pat)? {pp == len(pat)}")
print(f"return pt - pp = {pt} - {pp} = {pt - pp}")
print(f"확인: txt[{pt-pp}:{pt}] = '{txt[pt-pp:pt]}'")
print(f"\n총 비교 횟수: {step}회 (len(txt)={len(txt)}, len(pat)={len(pat)})")

### 4. 🟡 [손] 실패하는 경우 추적

이번엔 **못 찾는** 경우야.

```
  txt  A B C D E F
  pat  X Y Z
```

- 몇 번 비교하고 끝날까? **예측**해봐.
- 종료 시점의 `pt`와 `pp`는? `while` 조건 중 **어느 쪽**이 먼저 깨져?
- `return` 문의 삼항 연산자에서 **`else -1`** 로 가는 이유를 설명해봐.

그리고 이번엔 **거의 다 맞다가 실패**하는 경우:

```
  txt  A A A A A B
  pat  A A A A A A
```

- 이건 몇 번 비교할까? 위의 `XYZ` 경우보다 **많을까 적을까?**
- 매 시작 위치에서 몇 글자씩 맞다가 틀리지? 이게 브루트 포스의 **최악 시나리오**야.

*(예측을 적은 뒤 실행)*

In [ ]:
def bf_count(txt, pat):
    """비교 횟수를 함께 반환"""
    pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return (pt - pp if pp == len(pat) else -1), c, pt, pp

cases = [
    ("ABCDEF", "XYZ", "첫 글자부터 항상 불일치"),
    ("AAAAAB", "AAAAAA", "거의 맞다가 실패 (최악)"),
    ("ABABCDEFGHA", "ABC", "교재 예제"),
]
for t, p, label in cases:
    r, c, pt, pp = bf_count(t, p)
    n, m = len(t), len(p)
    print(f"{label}")
    print(f"  txt='{t}' (n={n}), pat='{p}' (m={m})")
    print(f"  → 결과 {r}, 비교 {c}회 | 종료 시 pt={pt}, pp={pp}")
    print(f"     (n={n}, n*m={n*m}) — {'pt가 끝에 도달' if pt==n else 'pp가 끝에 도달'}\n")

### 5. 🟡 [손] 겹치는 패턴

```
  txt  A B A B A B A
  pat  A B A
```

- 이 패턴은 텍스트 안에 **몇 번** 나올까? 위치를 전부 적어봐. (겹치는 것 포함!)
- `bf_match`는 그 중 **어느 것**을 반환할까? 왜?
- 그럼 **모든 위치를 다 찾으려면** 어떻게 고쳐야 할까? (12번에서 직접 구현할 거야)
- 💡 미리 보기: `'ABABABA'.count('ABA')` 는 **2**를 반환해. 실제 등장은 3번인데! 왜 그런지는 15번에서.

*(답을 적은 뒤 실행)*

In [ ]:
t, p = "ABABABA", "ABA"
print(f"txt = {t}, pat = {p}\n")
print("모든 시작 위치를 직접 확인:")
for i in range(len(t) - len(p) + 1):
    seg = t[i:i+len(p)]
    print(f"  i={i}: txt[{i}:{i+len(p)}] = '{seg}'  {'✅ 일치' if seg == p else ''}")

print(f"\nbf_match 결과: {bf_match(t, p)}  ← 맨 앞 하나만 반환")
print(f"str.count 결과: {t.count(p)}  ← 실제 등장은 3번인데?! (15번에서)")

---
# 🔑 PART 3 — `pt - pp` 불변식 (6~8번)

> **오늘의 심장.** 코드는 다 이해했는데 `return pt - pp` 만 찜찜하다면, 여기서 풀린다.

### 6. 🟡 [손] 🔥 `pt - pp` 열을 세로로 읽어라

3번에서 만든 추적표에 **한 열을 추가**해봐.

```
  txt  A B A B C D E F G H A       pat  A B C
```

| 단계 | pt | pp | **pt - pp** | 결과 |
|---|---|---|---|---|
| ① | 0 | 0 | **0** | 일치 |
| ② | 1 | 1 | ① | 일치 |
| ③ | 2 | 2 | ② | 불일치 |
| ④ | 1 | 0 | ③ | 불일치 |
| ⑤ | 2 | 0 | ④ | 일치 |
| ⑥ | 3 | 1 | ⑤ | 일치 |
| ⑦ | 4 | 2 | ⑥ | 일치 |

**`pt - pp` 열만 세로로 읽어봐.** 어떤 패턴이 보여?

- 값이 **변하지 않는 구간**은 언제야?
- 값이 **1 증가하는 순간**은 언제야?
- 그럼 `pt - pp` 는 대체 **무엇을 뜻하는 숫자**일까? 3번의 `show_align` 출력에서 패턴이 그려진 위치와 비교해봐.

*(패턴을 찾은 뒤 실행)*

In [ ]:
txt, pat = TXT, PAT
pt = pp = 0
rows = []
while pt != len(txt) and pp != len(pat):
    matched = txt[pt] == pat[pp]
    rows.append((pt, pp, pt - pp, "일치" if matched else "불일치"))
    if matched: pt += 1; pp += 1
    else: pt = pt - pp + 1; pp = 0

print("단계 | pt | pp | pt-pp | 결과   | 패턴이 놓인 위치")
print("-----+----+----+-------+--------+" + "-"*24)
for k, (a, b, d, r) in enumerate(rows, 1):
    bar = " " * d + pat
    print(f"  {k}  | {a:2d} | {b:2d} |   {d}   | {r:6s} | {bar}")

print(f"\npt-pp 열: {[d for _,_,d,_ in rows]}")
print("→ 대조 중에는 고정, 실패하면 정확히 1 증가")

### 7. 🔴 [설명] 🔥 불변식을 증명하라

6번에서 발견한 걸 이제 **코드로 증명**해보자.

> **주장: `pt - pp` 는 항상 "지금 시도 중인 대조의 시작 위치"다.**

이 주장이 참이려면 **두 갈래 모두**에서 성립해야 해.

**① 일치할 때**
```python
pt += 1
pp += 1
```
- 새로운 `pt - pp` = `(pt+1) - (pp+1)` = ①________
- 즉 값이 ②________
- **왜 이게 옳아?** 대조를 계속하는 중이니까 시작 위치는 ③________ 야 해.

**② 불일치할 때**
```python
pt = pt - pp + 1
pp = 0
```
- 새로운 `pt - pp` = `(pt - pp + 1) - 0` = ④________
- 즉 기존 값에서 ⑤________
- **왜 이게 옳아?** 이번 시도가 실패했으니 시작 위치를 ⑥________ 야 해.

**결론**: 이 한 줄 `pt = pt - pp + 1` 이 곧 **"패턴을 오른쪽으로 1칸 민다"** 는 동작 그 자체야. 교재 306p의 ⓑ, ⓒ 설명("패턴을 오른쪽으로 1칸 밉니다")이 코드에서는 이 형태로 나타난 거지.

**그리고 종료 시점**
- 루프가 `pp == len(pat)` 로 끝났다 = 패턴을 **끝까지 다 맞췄다**
- 그 시점의 `pt`는 매칭 구간의 **⑦________**, `pp`는 **⑧________**
- 그래서 `시작 = pt - pp` ✅

💡 이런 걸 **불변식(invariant)** 이라고 해. 23일차 병합 정렬의 `k ≤ i`, 24일차 힙의 "a[left] 이외는 모두 힙"과 같은 계열이야. **좋은 알고리즘에는 이런 게 하나씩 숨어 있어.**

*(답을 적은 뒤 실행)*

In [ ]:
print("불변식 자동 검증: pt-pp 가 항상 '현재 대조 시작 위치'인가?")
random.seed(0)
violations = 0
checks = 0
for _ in range(2000):
    t = ''.join(random.choice('AB') for _ in range(random.randint(1, 15)))
    p = ''.join(random.choice('AB') for _ in range(random.randint(1, 4)))
    pt = pp = 0
    start = 0                     # 우리가 따로 추적하는 '진짜 시작 위치'
    while pt != len(t) and pp != len(p):
        checks += 1
        if pt - pp != start:      # 불변식 위반?
            violations += 1
        if t[pt] == p[pp]:
            pt += 1; pp += 1
        else:
            pt = pt - pp + 1; pp = 0
            start += 1            # 실패 → 시작 위치 1칸 이동
print(f"  총 {checks:,}회 검사 중 위반 {violations}회")
print(f"  → 불변식 성립 {'✅' if violations == 0 else '❌'}")

print("\n결과 정확성도 확인 (naive와 비교)")
fail = 0
for _ in range(5000):
    t = ''.join(random.choice('AB') for _ in range(random.randint(0, 15)))
    p = ''.join(random.choice('AB') for _ in range(random.randint(1, 4)))
    if bf_match(t, p) != naive(t, p): fail += 1
print(f"  랜덤 5000회 불일치: {fail}회")

### 8. 🟡 [실험] 시간 복잡도 — "되돌아감"의 대가

2번에서 본 교재의 지적: **"이미 검사한 위치를 기억하지 못하므로 효율이 좋지 않습니다."**

`pt`가 되돌아간다는 건 **같은 글자를 여러 번 비교한다**는 뜻이야. 얼마나 손해일까?

**예측해봐** (텍스트 길이 n, 패턴 길이 m):
- 최선의 경우 비교 횟수는? ①____
- 최악의 경우는? ②____
- 최악이 되는 입력의 **모양**은? ③________________

*(예측을 적은 뒤 실행)*

In [ ]:
def bf_cmp(txt, pat):
    pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return c

print("n을 키워가며 비교 횟수 관찰 (최악 패턴: 'AAA...AB' 안에서 'AAAB' 찾기)")
print("   n  |  m  | 비교횟수 |  n*m   | n+m")
print("------+-----+----------+--------+------")
for n in (50, 100, 200, 400):
    m = 5
    t = 'A' * n
    p = 'A' * (m-1) + 'B'
    c = bf_cmp(t, p)
    print(f" {n:4d} | {m:3d} | {c:8d} | {n*m:6d} | {n+m:4d}")

print("\n→ 비교 횟수가 n+m 이 아니라 n*m 쪽에 가깝다 → O(n·m)")

print("\n[최선/최악 비교]")
for t, p, lbl in (('B'*1000, 'A', '최선 (첫 글자부터 불일치)'),
                  ('A'*1000, 'A'*10 + 'B', '최악 (거의 맞다가 실패)')):
    print(f"  {lbl:24s}: 비교 {bf_cmp(t,p):6,d}회 (n={len(t)}, m={len(p)})")

print("\n💡 실무 텍스트에서는 대부분 첫 글자에서 바로 틀려서 평균은 O(n)에 가깝다.")
print("   하지만 'AAAA...' 같은 반복 문자열에서는 최악 O(n·m)이 실제로 발생한다.")

---
# 💻 PART 4 — 코드 구현 (9~12번)

### 9. 🟢 [빈칸] `bf_match` 직접 구현

3~7번에서 뜯어본 걸 조립해.

**기대 출력**
```
2
-1
랜덤 1000회 검증: 실패 0회
```

In [ ]:
def my_bf_match(txt: str, pat: str) -> int:
    """브루트 포스법으로 문자열 검색"""
    pt = 0        # txt를 따라가는 커서
    pp = 0        # pat를 따라가는 커서

    while ___ and ___:                # ①② 언제까지 반복?
        if ___:                       # ③ 비교 조건
            pt += 1
            pp += 1
        else:
            pt = ___                  # ④ 시작 위치를 1칸 밀기 (7번!)
            pp = ___                  # ⑤

    return ___ if ___ else -1         # ⑥⑦ 성공 시 반환값, 성공 판정


print(my_bf_match('ABABCDEFGHA', 'ABC'))
print(my_bf_match('ABCDEF', 'XYZ'))

fail = 0
for _ in range(1000):
    t = ''.join(random.choice('ABC') for _ in range(random.randint(0, 20)))
    p = ''.join(random.choice('ABC') for _ in range(random.randint(1, 5)))
    if my_bf_match(t, p) != naive(t, p): fail += 1
print(f"랜덤 1000회 검증: 실패 {fail}회")

### 10. 🔴 [디버깅] 🔥 `pt = pt + 1` 로 바꾸면

```python
else:
    pt = pt - pp + 1     # 교재
    pp = 0
```
를 이렇게 바꿔보자:
```python
else:
    pt = pt + 1          # 🐛 -pp 를 뺐다
    pp = 0
```

- 7번의 불변식으로 생각해봐. 새 `pt - pp` = `(pt + 1) - 0` = `pt + 1` 이야. 원래는 `(pt - pp) + 1` 이었지.
- **`pp`가 0일 때는 둘이 같아.** 그럼 언제 달라져?
- 시작 위치가 **몇 칸** 밀리게 될까? 정확히는 `pp + 1` 칸이야. 왜 그런지 설명해봐.
- **결과는 어떻게 나올까?** 에러가 날까, 조용히 틀릴까? 어떤 입력에서 틀릴까?
- 💡 이건 **"패턴을 여러 칸 밀어버리는" 버그**인데, 재밌게도 07-2 KMP는 **의도적으로** 여러 칸을 밀어. 차이가 뭘까? (KMP는 밀어도 되는 만큼만 안전하게 민다)

*(예측을 적은 뒤 실행)*

In [ ]:
def bf_bug(txt, pat):
    """pt = pt + 1 로 바꾼 버전"""
    pt = pp = 0
    while pt != len(txt) and pp != len(pat):
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt + 1; pp = 0        # 🐛
    return pt - pp if pp == len(pat) else -1

tests = [('ABABCDEFGHA','ABC'), ('AABAABAAA','AABAAA'),
         ('ABABAB','ABAB'), ('SAKURA','KUR'), ('BBAABBB','BA')]
print("  txt            pat        정상  버그")
print("  " + "-"*44)
for t, p in tests:
    ok, bug = bf_match(t, p), bf_bug(t, p)
    mark = "" if ok == bug else "   ❌"
    print(f"  {t:14s} {p:10s} {ok:4d} {bug:5d}{mark}")

random.seed(3)
fail = 0; example = None
for _ in range(3000):
    t = ''.join(random.choice('AB') for _ in range(random.randint(1, 14)))
    p = ''.join(random.choice('AB') for _ in range(random.randint(1, 4)))
    if bf_bug(t, p) != naive(t, p):
        fail += 1
        if example is None: example = (t, p, bf_bug(t,p), naive(t,p))
print(f"\n랜덤 3000회 중 오답 {fail}회 ({fail/30:.1f}%)")
print(f"예: txt='{example[0]}', pat='{example[1]}' → 버그 {example[2]}, 정답 {example[3]}")
print("\n→ 에러 없이 '못 찾았다(-1)'고 조용히 거짓말한다 🔥")

### 11. 🟡 [디버깅] 종료 판정을 `pt`로 하면

```python
return pt - pp if pp == len(pat) else -1
```
를 이렇게 바꾸면:
```python
return pt - pp if pt == len(txt) else -1     # 🐛
```

- `while` 루프는 **두 가지 이유**로 빠져나올 수 있어. 각각 뭐야?
- 성공을 뜻하는 건 그중 **어느 쪽**이지?
- `pt == len(txt)` 로 판정하면 어떤 오답이 나올까? **두 방향**으로 생각해봐:
  - 실제로 찾았는데 `-1`이 나오는 경우
  - 못 찾았는데 엉뚱한 숫자가 나오는 경우
- 25일차 15번(`max([])`), 22일차 5번과 마찬가지로 **"판정 기준을 잘못 고르면 조용히 틀린다"** 는 패턴이야.

*(예측을 적은 뒤 실행)*

In [ ]:
def bf_bug2(txt, pat):
    """종료 판정을 pt로"""
    pt = pp = 0
    while pt != len(txt) and pp != len(pat):
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return pt - pp if pt == len(txt) else -1    # 🐛

tests = [('ABABCDEFGHA','ABC'), ('AABAABAAA','AABAAA'),
         ('ABABAB','ABAB'), ('SAKURA','KUR'), ('ABCDEF','XYZ')]
print("  txt            pat        정상  버그   문제")
print("  " + "-"*56)
for t, p in tests:
    ok, bug = bf_match(t, p), bf_bug2(t, p)
    if ok == bug: note = ""
    elif ok != -1 and bug == -1: note = "찾았는데 못 찾았다고 함"
    else: note = "못 찾았는데 숫자를 반환 🔥"
    print(f"  {t:14s} {p:10s} {ok:4d} {bug:5d}   {note}")

print("\n[왜?] 루프 탈출 조건은 두 가지")
print("  · pp == len(pat)  → 패턴을 끝까지 맞췄다 = 성공 ✅")
print("  · pt == len(txt)  → 텍스트를 다 훑었다 = 실패 (보통)")
print("  단, 패턴이 텍스트 맨 끝에서 매칭되면 두 조건이 동시에 참이 되기도 한다.")

### 12. 🔴 [구현] 모든 위치 찾기

5번에서 봤듯 `bf_match`는 **맨 앞 하나만** 반환해. 겹치는 것까지 **전부** 찾도록 확장해봐.

```
txt = "ABABABA", pat = "ABA"  →  [0, 2, 4]
```

**두 가지 방법이 있어:**
1. `bf_match`를 반복 호출하면서 시작 위치를 옮기기 (`txt[i:]`를 넘기고 오프셋 보정)
2. `bf_match` 내부를 고쳐서, 매칭 성공 시 기록하고 **1칸만 밀어** 계속 진행

2번이 더 효율적이야 (문자열 슬라이싱 복사가 없으니까). 2번으로 짜봐.

**힌트**: `pp == len(pat)` 가 되는 순간 `pt - pp` 를 기록하고, 마치 실패한 것처럼 시작 위치를 1칸 밀면 돼.

**기대 출력**
```
[0, 2, 4]
[0, 1, 2]
[]
랜덤 500회 검증: 실패 0회
```

In [ ]:
def bf_match_all(txt: str, pat: str) -> list:
    """겹침을 포함해 pat가 등장하는 모든 시작 인덱스를 반환"""
    result = []
    if len(pat) == 0: return result
    pt = pp = 0
    while pt != len(txt):
        if txt[pt] == pat[pp]:
            pt += 1
            pp += 1
            if pp == len(pat):            # 매칭 성공!
                result.append(___)        # ① 시작 위치 기록
                pt = ___                  # ② 1칸만 밀어서 계속 (겹침 허용)
                pp = ___                  # ③
        else:
            pt = pt - pp + 1
            pp = 0
    return result


print(bf_match_all("ABABABA", "ABA"))
print(bf_match_all("AAAA", "AA"))
print(bf_match_all("ABCDEF", "XYZ"))

def naive_all(t, p):
    return [i for i in range(len(t)-len(p)+1) if t[i:i+len(p)] == p]

fail = 0
for _ in range(500):
    t = ''.join(random.choice('AB') for _ in range(random.randint(1, 20)))
    p = ''.join(random.choice('AB') for _ in range(random.randint(1, 4)))
    if bf_match_all(t, p) != naive_all(t, p): fail += 1
print(f"랜덤 500회 검증: 실패 {fail}회")

---
# 🐍 PART 5 — 파이썬이 이미 갖고 있는 것 (13~17번)

> 교재 308~309p 보충수업 7-1.
> 22일차 `sorted()`, 24일차 `heapq`, 25일차와 같은 자리야 — **손으로 짠 뒤에 표준 도구를 만난다.**

### 13. 🟢 [설명] 멤버십 연산자 `in`

교재 308p:
```
ptn in txt        →  ptn은 txt에 포함되어 있습니까?
ptn not in txt    →  ptn은 txt에 포함되어 있지 않습니까?
```

- `'ABC' in 'ABABCDEFGHA'` 는 뭘 반환해? 타입은?
- 교재의 결정적 한 문장: **"이 방법은 어떤 문자열이 다른 문자열 안에 포함되어 있는지 판단할 수는 있지만 그 위치는 알지 못합니다."**
  → 1번에서 나눈 **두 가지 질문** 중 `in`은 어느 쪽에만 답하지?
- 그럼 `in`은 언제 쓰고 `find`는 언제 써?
- 3일차에 배운 `in`은 리스트/집합에서도 썼지. 문자열에서의 `in`은 뭐가 다를까? (힌트: `'BC' in 'ABCD'` vs `'BC' in ['A','B','C']`)

*(답을 적은 뒤 실행)*

In [ ]:
txt = 'ABABCDEFGHA'
print(f"'ABC' in txt      → {'ABC' in txt}  (타입: {type('ABC' in txt).__name__})")
print(f"'XYZ' in txt      → {'XYZ' in txt}")
print(f"'XYZ' not in txt  → {'XYZ' not in txt}")
print("\n위치는? → in 은 알려주지 않는다. bool 하나뿐.")

print("\n[리스트의 in 과 다르다]")
print(f"  'BC' in 'ABCD'            → {'BC' in 'ABCD'}   (부분 문자열 검색)")
print(f"  'BC' in ['A','B','C','D'] → {'BC' in ['A','B','C','D']}  (원소가 정확히 같은지)")
print("  → 문자열의 in 은 '연속된 부분'을 찾고, 리스트의 in 은 '원소 하나'를 찾는다")

### 14. 🟢 [설명] find / rfind / index / rindex

교재 309p가 정리한 네 함수야.

```
str.find(sub[, start[, end]])
str.rfind(sub[, start[, end]])
str.index(sub[, start[, end]])
str.rindex(sub[, start[, end]])
```

**표를 채워봐.**

| 함수 | 어느 쪽부터 찾나 | 반환 | 못 찾으면 |
|---|---|---|---|
| `find` | 앞 | 시작 인덱스 | ① |
| `rfind` | ② | ③ | ④ |
| `index` | 앞 | 시작 인덱스 | ⑤ |
| `rindex` | 뒤 | 시작 인덱스 | ⑥ |

- 접두사 `r` 은 무슨 뜻일까? (교재 PDF 필기에도 적어놨듯 **"뒤에서부터 찾겠다"**)
  같은 `r` 접두사를 쓰는 다른 문자열 메서드도 있어. 뭐가 있지?
- **`find`와 `index`의 차이는 실패 처리뿐**이야. 언제 어느 걸 써야 할까?
- 🔥 `rfind`가 반환하는 건 **"마지막 등장의 시작 인덱스"** 야. `find.py`/`index.py` 코드의 출력문은 이걸 "맨 끝 인덱스"라고 부르는데, 오해의 소지가 있어. 진짜 **끝** 인덱스를 구하려면?

*(답을 적은 뒤 실행)*

In [ ]:
t, p = 'ABABABA', 'ABA'
print(f"txt = '{t}', ptn = '{p}'\n")
print(f"  find   : {t.find(p)}    ← 맨 앞 등장의 시작")
print(f"  rfind  : {t.rfind(p)}    ← 맨 뒤 등장의 시작 (끝이 아님!)")
print(f"  진짜 끝 인덱스: rfind + len(ptn) - 1 = {t.rfind(p) + len(p) - 1}")
print(f"  확인: t[{t.rfind(p)}:{t.rfind(p)+len(p)}] = '{t[t.rfind(p):t.rfind(p)+len(p)]}'")

print("\n[못 찾을 때]")
print(f"  find('ZZ')  → {t.find('ZZ')}")
try:
    t.index('ZZ')
except ValueError as e:
    print(f"  index('ZZ') → ValueError: {e}")

print("\n[start, end 인자 — 슬라이스 표기와 같은 규칙]")
print(f"  t.find('ABA')       = {t.find('ABA')}")
print(f"  t.find('ABA', 1)    = {t.find('ABA', 1)}    ← 인덱스 1부터 찾기")
print(f"  t.find('ABA', 1, 5) = {t.find('ABA', 1, 5)}    ← t[1:5] 범위 안에서")
print(f"  t.find('ABA', 5)    = {t.find('ABA', 5)}   ← 범위 안에 없음")

### 15. 🟡 [실험] 🔥 `count`는 겹치는 걸 세지 않는다

`find.py` / `index.py` 코드는 둘 다 `count`로 시작해:

```python
c = txt.count(ptn)
if c == 0: ...
elif c == 1: ...
else: ...
```

5번에서 예고한 함정이 여기 있어.

```
txt = "ABABABA", ptn = "ABA"
```
- 실제 등장 위치는 `[0, 2, 4]` — **3번**
- 그런데 `count` 는 **2**를 반환해

**왜일까?**
- `count`는 하나 찾으면 **그 뒤부터** 다시 찾아. 0에서 찾고 → 다음은 인덱스 3부터 → 4에서 찾음. 인덱스 2는 **건너뛰어**.
- 우리가 12번에서 짠 `bf_match_all`은 **1칸만** 밀어서 겹침도 다 찾지.

**표를 채워봐:**

| txt | ptn | `count` | 겹침 포함 실제 개수 |
|---|---|---|---|
| `"ABABABA"` | `"ABA"` | 2 | 3 |
| `"AAAA"` | `"AA"` | ① | ② |
| `"AAAAA"` | `"AA"` | ③ | ④ |

- 이 코드에서 `count`가 **0/1/2+ 판정에만** 쓰여서 큰 문제는 안 생겨. 하지만 언제 물릴까?
- 12번의 `bf_match_all`에서 `pt = pt - pp + 1` 대신 `pt = pt` (그대로 두고 `pp = 0`만) 로 하면 `count`와 같은 동작이 될까? 어떻게 고쳐야 count와 같아질까?

*(답을 적은 뒤 실행)*

In [ ]:
def find_all_overlap(t, p):
    """겹침 포함 (bf_match_all과 같은 규칙)"""
    res = []; i = t.find(p)
    while i != -1:
        res.append(i); i = t.find(p, i + 1)      # ← 1칸만 이동
    return res

def find_all_nonoverlap(t, p):
    """겹침 제외 (count와 같은 규칙)"""
    res = []; i = t.find(p)
    while i != -1:
        res.append(i); i = t.find(p, i + len(p))  # ← 패턴 길이만큼 이동
    return res

print(" txt      ptn   count  겹침포함        겹침제외")
print("-" * 56)
for t, p in (("ABABABA","ABA"), ("AAAA","AA"), ("AAAAA","AA"), ("ABCABC","ABC")):
    print(f" {t:9s}{p:5s} {t.count(p):5d}  {str(find_all_overlap(t,p)):15s} {find_all_nonoverlap(t,p)}")

print("\n→ count 는 '겹침 제외' 규칙과 일치한다")
print("→ 차이는 딱 한 글자: find(p, i+1) vs find(p, i+len(p))")

### 16. 🟢 [실험] `startswith` / `endswith`

교재 309p "with 계열 함수":
```
str.startswith(prefix[, start[, end]])
str.endswith(suffix[, start[, end]])
```

- 이 둘은 **어떤 질문**에 답하는 함수야? `find`와 뭐가 달라?
- `txt.startswith(ptn)` 은 `txt.find(ptn) == 0` 과 항상 같은 결과일까? 반례가 있을까?
- `txt.endswith(ptn)` 을 `rfind`로 표현하면? (⚠️ 조심해서 써봐)
- 실무에서 이게 쓰이는 곳: 파일 확장자 검사(`.endswith('.py')`), URL 프로토콜 검사(`.startswith('https://')`) 등. 왜 `find`보다 이게 나을까?

*(답을 적은 뒤 실행)*

In [ ]:
t = 'ABABCDEFGHA'
print(f"txt = '{t}'\n")
print(f"  startswith('ABA') → {t.startswith('ABA')}")
print(f"  startswith('BAB') → {t.startswith('BAB')}")
print(f"  endswith('GHA')   → {t.endswith('GHA')}")
print(f"  endswith('ABA')   → {t.endswith('ABA')}")

print("\n[find로 흉내내기]")
p = 'ABA'
print(f"  t.startswith('{p}')  = {t.startswith(p)}")
print(f"  t.find('{p}') == 0   = {t.find(p) == 0}   ← 같은 결과")
print(f"  t.endswith('{p}')    = {t.endswith(p)}")
print(f"  t.rfind('{p}') == len(t)-len(p) = {t.rfind(p) == len(t)-len(p)}   ← 같은 결과")

print("\n[튜플로 여러 개 한 번에]")
files = ['main.py', 'note.ipynb', 'data.csv', 'run.sh']
for fn in files:
    print(f"  {fn:12s} 파이썬 관련? {fn.endswith(('.py', '.ipynb'))}")

print("\n[왜 find보다 나은가]")
print("  · 의도가 코드에 그대로 드러난다 (== 0 같은 비교가 필요 없음)")
print("  · 튜플로 여러 후보를 한 번에 검사할 수 있다")
print("  · 문자열 전체를 훑지 않고 앞/뒤만 보므로 더 빠르다")

### 17. 🟡 [실험] 우리 코드 vs `str.find`

24일차 `heapq`, 25일차와 같은 마무리야 — **손으로 짠 것과 표준 라이브러리의 격차를 재본다.**

**예측해봐** (최악 패턴, n = 200,000):
- `bf_match` 와 `str.find` 의 속도 차이는 몇 배쯤일까? ①____
- 둘 다 브루트 포스라면 차이가 클까 작을까?

*(예측을 적은 뒤 실행)*

In [ ]:
txt_big = 'A' * 200000 + 'B'
pat_big = 'A' * 50 + 'B'

t0 = time.perf_counter(); r1 = bf_match(txt_big, pat_big); e1 = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = txt_big.find(pat_big);      e2 = time.perf_counter() - t0

print(f"최악 패턴: n = {len(txt_big):,}, m = {len(pat_big)}")
print(f"  bf_match (우리 구현) : {e1:8.4f}s → {r1}")
print(f"  str.find (CPython)   : {e2:8.6f}s → {r2}")
print(f"  → 약 {e1/e2:,.0f}배 차이")

print("\n[왜 이렇게 차이가 나나]")
print("  1) C 구현 vs 파이썬 인터프리터 (상수 차이, 수십 배)")
print("  2) 알고리즘 자체가 다름 — CPython의 find는 브루트 포스가 아니라")
print("     Two-way 알고리즘(Boyer-Moore/KMP 아이디어 결합)을 쓴다")
print("     → 실패해도 여러 칸을 건너뛴다 = 07-2, 07-3에서 배울 그것!")

print("\n[평범한 텍스트 + 끝까지 못 찾는 경우]")
random.seed(1)
txt_s = ''.join(random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ') for _ in range(200000))
pat_s = 'ZZZZZZ'                      # 사실상 등장하지 않는 패턴
t0 = time.perf_counter(); r3 = bf_match(txt_s, pat_s); e3 = time.perf_counter() - t0
t0 = time.perf_counter(); r4 = txt_s.find(pat_s);      e4 = time.perf_counter() - t0
print(f"  bf_match: {e3:.4f}s → {r3}")
print(f"  str.find: {e4:.6f}s → {r4}")
print(f"  → 약 {e3/e4:,.0f}배 차이 (텍스트 전체를 훑는 경우)")


---
---

# ✅ 정답 & 해설

> ⚠️ **표를 직접 채운 뒤에 내려와.** 특히 3·6·7번은 손으로 써야 남아.

---

## 🔁 Remind

### R-1
- ① **일치하면 같이 +1, 불일치하면 `pt`는 되돌아가고 `pp`는 0으로 리셋**
- 병합 정렬의 `pa`/`pb`는 **매 반복 둘 중 하나만** 증가했지만, 오늘은 **둘 다 증가하거나 / 둘 다 리셋**되는 구조야. 이 "같이 움직인다"가 6~7번 불변식의 씨앗이야.

### R-2
- 선형 검색은 각 자리에서 **값 1개**를 비교. 문자열 검색은 각 자리에서 **최대 m개**(패턴 길이만큼)를 비교해.
- 그래서 O(n) → **O(n·m)**. 8번에서 실측해.
- "못 찾으면 -1" 규약은 `return pt - pp if pp == len(pat) else -1` 의 **`else -1`** 부분이야. 12일차부터 계속 이어져 온 관례지.

---

## 🎯 PART 1 해설

### 1. 텍스트와 패턴
- `'STRING'`에는 `I`, `N`이 **연속으로** 붙어 있어서 성공. `'QUEEN'`에는 `I`가 아예 없어서 실패.
- 보통 **텍스트가 훨씬 길어** (`n ≫ m`). 문서에서 단어를 찾는 상황을 떠올리면 돼.
- 두 가지 질문:
  - ① **포함되어 있는가?** (존재 여부, bool)
  - ② **어디에 있는가?** (위치, 인덱스)
- `in` 연산자는 **①에만** 답해 (13번).

### 2. "브루트 포스"
- 가능한 **모든 시작 위치를 하나도 빠짐없이** 시도하기 때문이야. 요령이 없어. 그래서 **단순법**이라고도 하지.
- **"되돌아간다"는 코드의 이 줄**:
```python
pt = pt - pp + 1     # ← pp만큼 되돌아가고 1칸 전진
```
`pp`가 2였다면 `pt`는 2칸 뒤로 갔다가 1칸 앞으로 → 결과적으로 **1칸 뒤로** 물러나. 이미 본 글자를 다시 보는 거지.
- 07-2 KMP는 이 줄에서 **`pt`를 되돌리지 않고 `pp`만 줄여.** 그게 핵심 차이야.

---

## ✍️ PART 2 해설

### 3. 그림 7-3 완전 재현

| 단계 | pt | pp | txt[pt] | pat[pp] | 일치? | 다음 |
|---|---|---|---|---|---|---|
| ① | 0 | 0 | A | A | ✅ | pt=1, pp=1 |
| ② | 1 | 1 | B | B | ✅ | pt=2, pp=2 |
| ③ | 2 | 2 | ① **A** | ② **C** | ❌ | pt=1, pp=0 |
| ④ | 1 | 0 | ③ **B** | ④ **A** | ⑤ **❌** | ⑥ **pt=2, pp=0** |
| ⑤ | ⑦ **2** | ⑧ **0** | ⑨ **A** | ⑩ **A** | ⑪ **✅** | ⑫ **pt=3, pp=1** |
| ⑥ | 3 | 1 | B | B | ✅ | pt=4, pp=2 |
| ⑦ | 4 | 2 | C | C | ✅ | pt=5, pp=3 |

**종료: `pt = 5`, `pp = 3` → `return 5 - 3 = 2`** ✅ (`txt[2:5] == 'ABC'`)

- ③→④에서 `pt`가 **2 → 1로 되돌아가지.** 교재가 지적한 "기억하지 못함"이 눈에 보이는 순간이야.
- 총 **7번** 비교. 텍스트 길이 11, 패턴 길이 3인데 7번이면 `n`(11)보다는 적고 `n·m`(33)보다도 적어. 실제 텍스트에선 대개 이렇게 운이 좋아.

---

### 4. 실패하는 경우

**`XYZ` (첫 글자부터 불일치)**
```
결과 -1, 비교 6회 | 종료 시 pt=6, pp=0
```
- 매 위치에서 첫 글자가 바로 틀리니 **한 번씩만** 비교하고 넘어가. `n`번 = 6회.
- 종료는 **`pt == len(txt)`** 쪽이 먼저 깨져서 일어나. `pp`는 0이라 `len(pat)`에 도달 못 했지.
- `pp != len(pat)` 이므로 `else -1`. 이게 "패턴을 끝까지 못 맞췄다"는 뜻이야.

**`AAAAAB` / `AAAAAA` (거의 맞다가 실패)**
```
결과 -1, 비교 11회 | 종료 시 pt=6, pp=0
```
- 매 시작 위치에서 **여러 글자를 맞추다가** 마지막에 틀려. 이게 브루트 포스의 최악 시나리오야 — 8번에서 본격적으로.

---

### 5. 겹치는 패턴

`"ABABABA"`에서 `"ABA"`는 **인덱스 0, 2, 4** — 총 **3번** 나와. 서로 겹치지.

- `bf_match`는 **맨 앞(0)** 만 반환해. 첫 매칭에서 `pp == len(pat)`가 되어 루프가 끝나니까.
- 모든 위치를 찾으려면 매칭 성공 후에도 **계속 진행**해야 해 → 12번.
- `count`가 2를 반환하는 건 **겹치는 걸 안 세기 때문** → 15번.

---

## 🔑 PART 3 해설

### 6. 🔥 `pt - pp` 열

| 단계 | pt | pp | **pt - pp** |
|---|---|---|---|
| ① | 0 | 0 | **0** |
| ② | 1 | 1 | ① **0** |
| ③ | 2 | 2 | ② **0** |
| ④ | 1 | 0 | ③ **1** |
| ⑤ | 2 | 0 | ④ **2** |
| ⑥ | 3 | 1 | ⑤ **2** |
| ⑦ | 4 | 2 | ⑥ **2** |

**세로로 읽으면: `0, 0, 0, 1, 2, 2, 2`**

- **변하지 않는 구간** = 대조가 진행 중일 때 (①②③, ⑤⑥⑦)
- **1 증가하는 순간** = 불일치가 나서 패턴을 민 직후 (③→④, ④→⑤)
- 그래서 `pt - pp` 는 **"지금 패턴이 놓여 있는 시작 위치"** 야. 실행 셀의 마지막 열(패턴 막대 그림)과 정확히 일치하지.

---

### 7. 🔥 불변식 증명

> **주장: `pt - pp` 는 항상 "지금 시도 중인 대조의 시작 위치"다.**

**① 일치할 때**
- 새 `pt - pp` = `(pt+1) - (pp+1)` = ① **`pt - pp`**
- 값이 ② **변하지 않는다**
- 왜 옳은가: 같은 시작 위치에서 대조를 **이어가는 중**이니 시작 위치는 ③ **그대로여야** 해. 두 커서가 **똑같이 1씩** 움직이는 게 이걸 보장하지.

**② 불일치할 때**
- 새 `pt - pp` = `(pt - pp + 1) - 0` = ④ **`(pt - pp) + 1`**
- 기존 값에서 ⑤ **정확히 1 증가**
- 왜 옳은가: 이번 시도가 실패했으니 시작 위치를 ⑥ **오른쪽으로 1칸 밀어야** 해. 교재 306p의 "패턴을 오른쪽으로 1칸 밉니다"가 코드에서는 이 모습이야.

**종료 시점**
- `pp == len(pat)` = 패턴을 끝까지 다 맞췄다
- `pt`는 ⑦ **매칭 구간의 바로 다음 칸**, `pp`는 ⑧ **매칭한 길이(= 패턴 길이)**
- 따라서 `시작 = 끝 다음 칸 - 길이 = pt - pp` ✅

**실측**: 랜덤 2,000개 케이스 × 모든 반복에서 **위반 0회.**

> 🔑 이게 오늘의 핵심이야. `pt = pt - pp + 1` 이라는 낯선 식이 사실은 **"시작 위치를 1 증가"** 라는 아주 단순한 의미였던 거지. 그리고 07-2 KMP는 바로 이 자리에서 **+1 대신 더 큰 값**을 넣어 — 그게 KMP의 전부야.

---

### 8. 시간 복잡도

**실측 (최악 패턴)**
```
   n  |  m  | 비교횟수 |  n*m   | n+m
  ----+-----+----------+--------+-----
   50 |   5 |      196 |    250 |   55
  100 |   5 |      396 |    500 |  105
  200 |   5 |      796 |   1000 |  205
  400 |   5 |     1596 |   2000 |  405
```
비교 횟수가 `n+m`이 아니라 **`n·m` 쪽에 비례**하며 자라. (여기선 약 `0.8·n·m`)

- ① 최선: **`n`번** (매번 첫 글자에서 바로 불일치)
- ② 최악: **`n·m`번** → **O(n·m)**
- ③ 최악의 모양: **패턴의 앞부분이 계속 맞다가 마지막에서 틀리는** 입력. `"AAAA...A"` 안에서 `"AAAB"` 찾기가 전형이야.

**실측 (최선 vs 최악, n=1000)**
```
최선 (첫 글자부터 불일치)  : 비교  1,000회
최악 (거의 맞다가 실패)    : 비교 10,901회
```

💡 실무 텍스트(영어 문장 등)는 첫 글자에서 대부분 걸러져서 **평균 O(n)에 가까워.** 그래서 브루트 포스도 실전에서 그럭저럭 쓸 만해. 하지만 DNA 서열(`AGCT`만 사용)이나 이진 데이터처럼 **알파벳이 작고 반복이 많은** 데이터에서는 최악이 실제로 터져.

---

## 💻 PART 4 해설

### 9. `my_bf_match`

```python
while pt != len(txt) and pp != len(pat):     # ①②
    if txt[pt] == pat[pp]:                   # ③
        pt += 1; pp += 1
    else:
        pt = pt - pp + 1                     # ④
        pp = 0                               # ⑤
return pt - pp if pp == len(pat) else -1     # ⑥⑦
```
출력: `2`, `-1`, 랜덤 1000회 실패 0회

**주의**: ①②는 `and`야. `or`로 쓰면 **둘 다 끝나야** 멈추므로 인덱스 에러가 나. 하나라도 끝나면 더 진행할 수 없으니 `and`가 맞아.

---

### 10. 🔥 `pt = pt + 1` 버그

**불변식으로 보면**: 새 `pt - pp` = `(pt + 1) - 0` = `pt + 1`.
원래는 `(pt - pp) + 1` 이었으니, 차이는 **`pp` 만큼**이야.

- **`pp == 0` 일 때는 둘이 같아.** 즉 첫 글자에서 바로 틀리는 경우엔 버그가 안 드러나.
- 몇 글자 맞춘 뒤 틀리면(`pp > 0`), 시작 위치가 **`pp + 1` 칸** 밀려. 원래 1칸만 밀어야 하는데 여러 칸을 건너뛰는 거지.
- 그래서 건너뛴 자리에 있던 정답을 **놓친다.**

**실측**
```
  txt            pat        정상  버그
  ABABCDEFGHA    ABC           2    -1   ❌
  AABAABAAA      AABAAA        3    -1   ❌
  ABABAB         ABAB          0     0
  SAKURA         KUR           2     2
  BBAABBB        BA            1    -1   ❌

랜덤 3000회 중 오답 268회 (8.9%)
```
- **에러 없이 "못 찾았다(-1)"고 조용히 거짓말**해. 9%만 틀리니 대충 테스트하면 통과할 수도 있어 — 22일차 10번(`pr -= 1`), 25일차 4번(음수 인덱스)과 같은 계열의 **조용한 버그**야.
- `SAKURA`/`KUR`처럼 **첫 매칭이 성공하는** 케이스는 멀쩡히 통과해서 더 위험해.

**KMP와의 차이** 💡: KMP도 여러 칸을 밀어. 하지만 **"패턴 자신의 구조를 미리 분석해서, 건너뛰어도 정답이 없다고 보장되는 만큼만"** 밀지. 이 버그는 **아무 근거 없이** 미는 거고. 같은 동작이라도 근거가 있느냐가 알고리즘과 버그를 가르는 거야.

---

### 11. 종료 판정을 `pt`로

**루프 탈출 조건 두 가지**
- `pp == len(pat)` → 패턴을 끝까지 맞췄다 = **성공**
- `pt == len(txt)` → 텍스트를 다 훑었다 = **보통 실패**

**실측**
```
  txt            pat        정상  버그   문제
  ABABCDEFGHA    ABC           2    -1   찾았는데 못 찾았다고 함
  AABAABAAA      AABAAA        3     3
  ABABAB         ABAB          0    -1   찾았는데 못 찾았다고 함
  SAKURA         KUR           2    -1   찾았는데 못 찾았다고 함
  ABCDEF         XYZ          -1     6   못 찾았는데 숫자를 반환 🔥
```

**두 방향으로 다 틀려**:
- 텍스트 중간에서 매칭되면 `pt < len(txt)` 라서 성공인데도 `-1`
- 못 찾고 텍스트를 다 훑으면 `pt == len(txt)` 라서 실패인데도 `pt - pp`(= 6) 반환

`AABAABAAA`/`AABAAA` 만 우연히 맞았는데, 이건 매칭이 **텍스트 맨 끝에서** 끝나 두 조건이 동시에 참이 된 경우야. **우연히 맞는 케이스가 있다**는 게 이 버그의 함정이지.

> 🔑 **성공을 정의하는 건 `pp`다.** `pt`는 "어디까지 봤나"일 뿐, "맞췄나"를 말해주지 않아.

---

### 12. 모든 위치 찾기

```python
if pp == len(pat):
    result.append(pt - pp)      # ① 불변식 그대로!
    pt = pt - pp + 1            # ② 실패했을 때와 똑같이 1칸 밀기
    pp = 0                      # ③
```
출력: `[0, 2, 4]`, `[0, 1, 2]`, `[]`, 랜덤 500회 실패 0회

- 매칭 성공 후에도 **실패했을 때와 똑같은 동작**(1칸 밀기)을 하는 게 포인트야. 그래야 겹치는 매칭도 잡히지.
- `while pt != len(txt)` 로 조건이 바뀐 것도 주목. 이제 `pp == len(pat)`가 종료 사유가 아니라 **기록 사유**가 됐거든.
- `if len(pat) == 0: return []` 가드가 없으면 `pat[0]`에서 IndexError. 25일차 15번의 빈 입력 방어와 같은 습관이야.

---

## 🐍 PART 5 해설

### 13. 멤버십 연산자

- `'ABC' in 'ABABCDEFGHA'` → `True`, 타입은 `bool`
- `in`은 **질문 ①(포함 여부)에만** 답해. 위치는 안 알려줘.
- **쓰임 구분**: 존재만 확인하면 `in` (의도가 명확하고 빠름), 위치가 필요하면 `find`.
- **리스트의 `in`과 다르다** 🔥
```
'BC' in 'ABCD'             → True    (연속된 부분 문자열)
'BC' in ['A','B','C','D']  → False   (원소가 정확히 'BC'인지)
```
문자열의 `in`만 **부분 문자열 검색**이야. 리스트는 원소 하나와의 일치를 봐. 3일차에 배운 `in`과 미묘하게 다르니 주의.

---

### 14. find / rfind / index / rindex

| 함수 | 방향 | 반환 | 못 찾으면 |
|---|---|---|---|
| `find` | 앞 | 시작 인덱스 | ① **-1** |
| `rfind` | ② **뒤** | ③ **마지막 등장의 시작 인덱스** | ④ **-1** |
| `index` | 앞 | 시작 인덱스 | ⑤ **ValueError** |
| `rindex` | 뒤 | 시작 인덱스 | ⑥ **ValueError** |

- `r` = **reverse**, 뒤에서부터. 같은 접두사: `rstrip`, `rsplit`, `rpartition`, `rindex`
- **`find` vs `index`**: 실패 처리만 달라. **없을 수도 있으면 `find`**(값으로 처리), **반드시 있어야 하면 `index`**(예외로 즉시 중단). 25일차 15번 `max([])` 와 같은 선택이야.
- 🔥 **`rfind`는 "맨 끝 인덱스"가 아니라 "마지막 등장의 시작 인덱스"**:
```
'ABABABA'.rfind('ABA') = 4      ← 시작 인덱스
진짜 끝 인덱스 = 4 + 3 - 1 = 6
```
`find.py`/`index.py`의 출력문 표현("맨 끝 인덱스")은 정확히는 "맨 뒤 등장의 시작 인덱스"라고 읽어야 해.
- **start, end 인자**는 슬라이스와 같은 규칙:
```
t.find('ABA', 1)    = 2     ← 인덱스 1부터
t.find('ABA', 1, 5) = 2     ← t[1:5] 범위 안에서
t.find('ABA', 5)    = -1
```

---

### 15. 🔥 `count`는 겹침을 안 센다

| txt | ptn | `count` | 겹침 포함 |
|---|---|---|---|
| `"ABABABA"` | `"ABA"` | 2 | `[0, 2, 4]` → 3 |
| `"AAAA"` | `"AA"` | ① **2** | ② `[0,1,2]` → **3** |
| `"AAAAA"` | `"AA"` | ③ **2** | ④ `[0,1,2,3]` → **4** |

**왜**: `count`는 매칭 후 **패턴 길이만큼** 건너뛰어. `find(p, i + len(p))` 와 같은 규칙이지. 우리 `bf_match_all`은 `find(p, i + 1)` 처럼 **1칸만** 밀어.

**차이는 딱 한 글자**:
```python
i = t.find(p, i + 1)        # 겹침 포함 (bf_match_all)
i = t.find(p, i + len(p))   # 겹침 제외 (count)
```

- `find.py`/`index.py`에선 `count`가 **0/1/2+ 판정에만** 쓰여서 결과가 안 틀려. 하지만 "이 단어가 몇 번 나오나"를 진짜 세야 할 때는 물려. DNA 서열 분석 같은 데선 치명적이지.
- 12번의 `bf_match_all`을 count와 같게 만들려면 `pt = pt - pp + 1` 대신 **`pt`는 그대로 두고 `pp = 0`** 으로 하면 돼 (이미 패턴 길이만큼 전진해 있으니까).

---

### 16. startswith / endswith

- 이 둘은 **위치가 특정된 포함 검사**야. `find`는 "어디든 있으면", 이건 "**시작/끝에** 있으면".
- `txt.startswith(ptn)` ≡ `txt.find(ptn) == 0` — 결과는 항상 같아.
- `txt.endswith(ptn)` ≡ `txt.rfind(ptn) == len(txt) - len(ptn)` — 이것도 같지만 훨씬 읽기 어렵지.
- **왜 `find`보다 나은가**:
  - **의도가 코드에 그대로 드러나** (`== 0` 같은 비교가 불필요)
  - **튜플로 여러 후보 검사**: `fn.endswith(('.py', '.ipynb'))`
  - 문자열 전체를 훑지 않고 **앞/뒤만** 보므로 빠름

실무 예: 파일 확장자 검사, URL 프로토콜 검사(`.startswith('https://')`), 접두사 필터링.

---

### 17. 우리 코드 vs `str.find`

**실측 (최악 패턴, n = 200,001, m = 51)**
```
bf_match (우리 구현) : 약 2.2s
str.find (CPython)   : 약 0.002s
→ 약 1,000배 차이
```

**평범한 텍스트에서 끝까지 못 찾는 경우 (n = 200,000)**
```
bf_match: 약 0.025s
str.find: 약 0.0002s
→ 약 100배 차이
```

**두 가지 원인**
1. **구현 언어**: C vs 파이썬 인터프리터 (수십~100배)
2. **알고리즘**: CPython의 `find`는 브루트 포스가 아니라 **Two-way 알고리즘**(Boyer-Moore와 KMP의 아이디어를 결합)을 써. 실패해도 **여러 칸을 건너뛰지.** → 최악 패턴에서 1,000배까지 벌어지는 이유가 이거야.

> 🔑 23일차 4번에서는 파이썬 루프의 상수 손해가 log n을 눌렀고, 오늘은 **알고리즘 우위까지 겹쳐서** 차이가 더 벌어졌어. 07-2, 07-3이 바로 그 "여러 칸 건너뛰기"를 배우는 절이야.

---

## 📌 핵심 3줄 요약

1. **브루트 포스는 "가능한 모든 시작 위치를 1칸씩 밀며 전부 시도"한다.** 실패하면 `pt`가 되돌아가므로 이미 본 글자를 다시 봐. 교재가 지적한 "이미 검사한 위치를 기억하지 못한다"가 코드에서는 `pt = pt - pp + 1` 이라는 한 줄이야.
2. **`pt - pp` 는 항상 "현재 대조의 시작 위치"다.** 일치하면 두 커서가 같이 움직여 값이 고정되고, 불일치하면 정확히 1 증가해. 그래서 종료 시 `pt - pp` 가 곧 답이야. `pt = pt + 1` 로 바꾸면 이 불변식이 깨져 시작 위치가 `pp+1` 칸씩 튀고, **에러 없이 정답을 놓친다.**
3. **성공 판정은 `pp`가 한다.** 루프는 `pt`가 끝나서도, `pp`가 끝나서도 빠져나오는데, 패턴을 다 맞췄다는 뜻은 오직 `pp == len(pat)` 뿐이야. `pt`로 판정하면 양방향으로 조용히 틀려.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 3, 9, 13, 14, 16번)**: 전원 필수
  - **3번 손 추적**이 오늘의 기본기. 교재 그림 7-3을 표로 옮기는 것부터
  - **14번 find/index 4형제**는 코테에서 매일 쓴다. `r` 접두사와 실패 처리 차이는 외워둘 것
- 🟡 **(R-2, 4, 5, 6, 8, 11, 15, 17번)**: 팀 목표선
  - **6번 `pt-pp` 열 세로 읽기** 🔥 — 여기서 "어?" 하는 순간이 오늘의 하이라이트
  - **15번 count 겹침 함정**은 실무에서 진짜 물리는 것
- 🔴 **(7, 10, 12번)**: 도전
  - **7번이 오늘 최고 난도** 🔥🔥 — 불변식을 두 갈래로 나눠 증명하면 `return pt - pp`가 완전히 이해돼
  - **10번**은 7번을 풀었으면 자연스럽게 보여. 못 풀었으면 7번으로 돌아갈 것
  - **12번**은 15번(count 겹침)과 짝이야
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 17번)

## 🔗 오늘 회수된 개념들

- **12일차 선형 검색** → 브루트 포스는 그 확장, "-1 반환" 규약도 그대로 (R-2)
- **18·21·23일차 투 포인터** (`pl`/`pr`, `pa`/`pb`/`pc`) → `pt`/`pp` 두 커서 (R-1)
- **23일차 `k ≤ i` 불변식 / 24일차 힙 가정** → 오늘의 `pt - pp` 불변식 (7번)
- **22일차 10번 / 25일차 4번 조용한 버그** → `pt = pt+1` 이 에러 없이 틀림 (10번)
- **25일차 15번 `max([])`** → `find` vs `index`의 실패 처리 선택 (14번)
- **22·24·25일차 표준 라이브러리 마무리** → `in`, `find` 계열, `with` 계열 (13~17번)
- **3일차 `in` 연산자** → 문자열의 `in`은 부분 문자열, 리스트는 원소 일치 (13번)

---

> **다음 진도 (27일차)**: 07-2 **KMP법**
>
> 오늘 7번에서 증명한 `pt = pt - pp + 1`(시작 위치 +1)이 KMP에서는 **`+1` 대신 더 큰 값**으로 바뀌어.
> 그 "더 큰 값"을 어떻게 안전하게 정하느냐가 KMP의 전부야. 힌트: **패턴 자기 자신의 반복 구조**를 미리 분석해둬.
> 오늘 10번에서 본 "근거 없이 여러 칸 밀면 정답을 놓친다"를, KMP는 **근거를 만들어서** 해결하는 거지.